# 03 Events and day-0 features

One table of candidate events (parliamentary elections, crises, planned cabinet changes and the two
ambiguous rows) with:
- **day 0** and **day +1** on each index's own trading calendar,
- **eligibility by history** (at least `MIN_HISTORY` earlier trading days, and a day +1),
- **overlap flags** among eligible events of the same country,
- the **political predictors** known on day 0.

The notebook also writes the political dates of every country (used later to clean estimation windows
and normal days), the sample flow and the list of missing predictors. It computes no returns and no
outcomes: the only use of prices here is the trading calendar.

Rules applied throughout:
- labels, dates and ids of `crises.csv` and `robustness_sets.csv` are never changed;
- ParlGov tables are joined on ids (`cabinet_id`, `election_id`, `previous_cabinet_id`,
  `previous_parliament_election_id`), never on names;
- every analysis choice is a constant imported from `helpers.py`; numbers typed below are data checks.
- Nothing in this notebook is random, so no seed is set.

In [1]:
import sys
from pathlib import Path

# On Nuvolos the kernel does not start in the notebook's folder: find code/ as 02_sample does.
roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import numpy as np
import pandas as pd

from helpers import (DATA, LR_UNKNOWN_FLAG, MAJORITY_SHARE, MIN_HISTORY, OVERLAP_FLAG, TICKERS, eligible_from,
                     load_cabinets, load_elections, load_prices, raw_file, save_derived)
from analysis import (ANALYSIS_GROUPS, CRISIS_LABELS, apply_robustness_set, calendar_position,
                      load_robustness_sets, overlap_flags, trading_calendar, valid_closes)

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

Working in /files/mywork/elections_project/code


## a) Events

### Hand-coded cabinet changes and robustness sets
`is_crisis` and the date columns are read as text, so that "1", "0", "ambiguous" and empty stay
exactly as coded (pandas would otherwise turn them into numbers or dates).

In [2]:
DATE_COLS = ["parlgov_start_date", "trigger_date", "alt_trigger_dates", "resignation_date"]
TEXT_COLS = ["is_crisis"] + DATE_COLS
crises = pd.read_csv(DATA / "crises.csv", dtype={c: str for c in TEXT_COLS})
crises[TEXT_COLS] = crises[TEXT_COLS].fillna("")
rob = load_robustness_sets()

assert len(crises) == 149 and crises["event_id"].is_unique
label_counts = crises["is_crisis"].value_counts().to_dict()
assert label_counts == {"1": 80, "0": 44, "ambiguous": 2, "": 23}, label_counts
coded = crises["is_crisis"] != ""
assert (crises.loc[coded, "trigger_date"] != "").all(), "a coded row has no trigger_date"
assert rob["event_id"].isin(crises["event_id"]).all(), "robustness_sets.csv lists an unknown event_id"
assert crises["country"].isin(list(TICKERS)).all()
assert (crises["event_id"] == crises["country"] + "_" + crises["parlgov_cabinet_id"].astype(str)).all()
assert set(rob["action"]) <= {"drop", "relabel"}
assert set(rob.loc[rob["action"] == "relabel", "new_is_crisis"]) <= set(CRISIS_LABELS)

# Every alternative trigger date of a robustness set is already a dated candidate of its row,
# so it is also a political date of the country (section c).
by_id = crises.set_index("event_id")
for _, r in rob[rob["new_trigger_date"] != ""].iterrows():
    row = by_id.loc[r["event_id"]]
    row_dates = set(row[DATE_COLS].str.split(";").explode())
    assert set(r["new_trigger_date"].split(";")) <= row_dates, r["event_id"]
print("crises.csv:", label_counts)

crises.csv: {'1': 80, '0': 44, '': 23, 'ambiguous': 2}


### ParlGov
Elections: parliamentary only, from `EVENTS_FROM`, 23 countries (`load_elections`). The party-level
rows of `view_election` are also needed for every parliamentary election of the 23 countries, including
those before 1997: they give seat shares, the previous parliament of the first events, and political
dates that fall in the estimation windows of the 1997-98 events.

In [3]:
elections = load_elections()
assert len(elections) == 174 and elections["election_id"].is_unique, len(elections)

el_party = pd.read_csv(raw_file("parlgov_view_election"), parse_dates=["election_date"])
el_party = el_party[(el_party["election_type"] == "parliament")
                    & el_party["country_name_short"].isin(list(TICKERS))]
assert not el_party.duplicated(["election_id", "party_id"]).any(), "two rows for one party in one election"
assert elections["election_id"].isin(el_party["election_id"]).all()

cab = load_cabinets()
cab1 = cab.drop_duplicates("cabinet_id").set_index("cabinet_id")   # one row per cabinet
per_cabinet = ["start_date", "cabinet_name", "election_id", "previous_cabinet_id", "election_seats_total",
               "country_name_short"]
assert (cab.groupby("cabinet_id")[per_cabinet].nunique(dropna=False) <= 1).all().all(), \
    "a cabinet-level column varies within a cabinet"

# Join check of the hand-coded rows: the new cabinet exists, in the same country.
ids = crises["parlgov_cabinet_id"]
assert ids.isin(cab1.index).all(), crises.loc[~ids.isin(cab1.index), "event_id"].tolist()
assert (cab1.loc[ids, "country_name_short"].to_numpy() == crises["country"].to_numpy()).all()
start_diff = crises[cab1.loc[ids, "start_date"].dt.strftime("%Y-%m-%d").to_numpy()
                    != crises["parlgov_start_date"].to_numpy()]
print("crises.csv rows whose parlgov_start_date differs from view_cabinet:", len(start_diff))
print(start_diff[["event_id", "parlgov_start_date"]].to_string(index=False) if len(start_diff) else "")

crises.csv rows whose parlgov_start_date differs from view_cabinet: 0



### Event rows
- Elections: incumbent = `view_election.previous_cabinet_id` (the cabinet in office at the vote);
  parliament for `enp` = the parliament elected in that election.
- Cabinet changes (`is_crisis` "1" -> crisis, "0" -> planned, "ambiguous" kept as group ambiguous;
  uncoded rows are not events): incumbent = the `previous_cabinet_id` of the new cabinet, i.e. the
  cabinet that fell; parliament = the incumbent's `election_id`. The new cabinet itself is never used,
  because it does not exist on day 0.
- One extra row (variant = set name) for each alternative day 0 that a robustness set needs. For a set
  that lists two dates (ISR_1128), day 0 comes from the first one; the event stays one event and the
  second date is only a political date.

In [4]:
GROUP_OF_LABEL = {**CRISIS_LABELS, "ambiguous": "ambiguous"}

el_ev = pd.DataFrame({
    "event_id": elections["country_name_short"] + "_E" + elections["election_id"].astype(str),
    "variant": "main",
    "country": elections["country_name_short"],
    "group": "election",
    "source_date": elections["election_date"],
    "incumbent_cabinet_id": elections["previous_cabinet_id"],
    "parliament_election_id": elections["election_id"],
    "previous_election_id": elections["previous_parliament_election_id"],   # used for lr_shift only
    "election_id": elections["election_id"],
    "parlgov_cabinet_id": np.nan,
})

cr = crises[coded]
incumbent = cr["parlgov_cabinet_id"].map(cab1["previous_cabinet_id"])
cr_ev = pd.DataFrame({
    "event_id": cr["event_id"],
    "variant": "main",
    "country": cr["country"],
    "group": cr["is_crisis"].map(GROUP_OF_LABEL),
    "source_date": pd.to_datetime(cr["trigger_date"], format="%Y-%m-%d"),
    "incumbent_cabinet_id": incumbent,
    "parliament_election_id": incumbent.map(cab1["election_id"]),
    "previous_election_id": np.nan,
    "election_id": np.nan,
    "parlgov_cabinet_id": cr["parlgov_cabinet_id"],
})

alt_rows = []
for _, r in rob[rob["new_trigger_date"] != ""].iterrows():
    row = cr_ev[cr_ev["event_id"] == r["event_id"]].copy()
    row["variant"] = r["set"]
    row["source_date"] = pd.to_datetime(r["new_trigger_date"].split(";")[0], format="%Y-%m-%d")
    alt_rows.append(row)

events = pd.concat([el_ev, cr_ev, *alt_rows], ignore_index=True)
for c in ["incumbent_cabinet_id", "parliament_election_id", "previous_election_id", "election_id",
          "parlgov_cabinet_id"]:
    events[c] = events[c].astype("Int64")
assert not events.duplicated(["event_id", "variant"]).any()
print(events.groupby(["variant", "group"]).size().to_string())

variant   group    
R_bundle  planned        2
main      ambiguous      2
          crisis        80
          election     174
          planned       44


## b) Trading calendar, day 0 and eligibility by history
The trading calendar of a country is the set of dates with a valid close after the stale-price rule
(`analysis.trading_calendar`; notebook 04 uses the same function, so day 0 is never a close that 04
removes). Day 0 is the first trading day on or after the trigger date (cabinet changes) or the election
date; day +1 is the next trading day. An event is eligible by history if day 0 has at least
`MIN_HISTORY` earlier trading days and day +1 exists.

In [5]:
prices = load_prices()
calendars = {c: trading_calendar(prices, c) for c in TICKERS}
removed = pd.Series({c: prices[TICKERS[c]].notna().sum() - len(calendars[c]) for c in TICKERS})
print("Closes removed by the stale-price rule, by country (non-zero only):")
print(removed[removed > 0].to_string() if (removed > 0).any() else "none")

events["day0_pos"] = np.nan
for c, idx in events.groupby("country").groups.items():
    events.loc[idx, "day0_pos"] = calendar_position(calendars[c], events.loc[idx, "source_date"])

def day_at(country, pos):
    cal = calendars[country]
    return cal[int(pos)] if pd.notna(pos) and pos < len(cal) else pd.NaT

events["day0"] = [day_at(c, p) for c, p in zip(events["country"], events["day0_pos"])]
events["day1"] = [day_at(c, p + 1) for c, p in zip(events["country"], events["day0_pos"])]

# eligible_from on the valid closes, i.e. on the same trading calendar as day 0
clean = pd.DataFrame({TICKERS[c]: valid_closes(prices, c) for c in TICKERS})
first_eligible = {c: eligible_from(clean, c, MIN_HISTORY) for c in TICKERS}
events["eligible_from"] = events["country"].map(first_eligible)
has_history = events["day0"] >= events["eligible_from"]          # False when either date is missing
has_day1 = events["day1"].notna()
assert (has_history == (events["day0_pos"] >= MIN_HISTORY)).all()
events["eligible"] = has_history & has_day1
events["ineligible_reason"] = np.select(
    [events["day0"].isna(), ~has_history, ~has_day1],
    ["no trading day on or after the source date",
     f"fewer than {MIN_HISTORY} earlier trading days",
     "no day +1 in the price file"], default="")

# day 0 is a trading day of the index, on or after its source date
assert events.loc[events["group"] == "election", "day0"].notna().all()
d0 = events.dropna(subset=["day0"])
assert (d0["day0"] >= d0["source_date"]).all()
assert all(d in calendars[c] for c, d in zip(d0["country"], d0["day0"]))
assert (events.loc[events["group"] == "election", "variant"] == "main").all()

print("\nEvents not eligible by history:")
print(events.loc[~events["eligible"], ["event_id", "variant", "group", "source_date", "eligible_from",
                                       "ineligible_reason"]].to_string(index=False))

Closes removed by the stale-price rule, by country (non-zero only):
AUS     2
ISR     2
ITA    20
EST     9
LTU     8

Events not eligible by history:
event_id variant    group source_date eligible_from                   ineligible_reason
DNK_E389    main election  1998-03-11    2018-01-18 fewer than 270 earlier trading days
DNK_E632    main election  2001-11-20    2018-01-18 fewer than 270 earlier trading days
DNK_E563    main election  2005-02-08    2018-01-18 fewer than 270 earlier trading days
DNK_E174    main election  2007-11-13    2018-01-18 fewer than 270 earlier trading days
DNK_E722    main election  2011-09-15    2018-01-18 fewer than 270 earlier trading days
DNK_E976    main election  2015-06-18    2018-01-18 fewer than 270 earlier trading days
EST_E239    main election  1999-03-07    2014-06-11 fewer than 270 earlier trading days
EST_E423    main election  2003-03-02    2014-06-11 fewer than 270 earlier trading days
 EST_E83    main election  2007-03-04    2014-06-11 fewer

Check against `02_sample`: it found 145 eligible elections with `eligible_from` on the frozen data.
The count is computed here on the stale-price calendar and also on the raw closes (all non-missing
closes, as `eligible_from` sees the raw price file), so that a difference can be traced to the
stale-price rule.

In [6]:
el_rows = events["group"] == "election"
n_eligible_el = int(events.loc[el_rows, "eligible"].sum())

raw_first = {c: eligible_from(prices, c, MIN_HISTORY) for c in TICKERS}
raw_day0 = pd.Series(pd.NaT, index=events.index)
for c, idx in events[el_rows].groupby("country").groups.items():
    raw_cal = prices[TICKERS[c]].dropna().index
    pos = calendar_position(raw_cal, events.loc[idx, "source_date"])
    raw_day0[idx] = [raw_cal[int(p)] if pd.notna(p) else pd.NaT for p in pos]
raw_eligible = el_rows & (raw_day0 >= events["country"].map(raw_first))
print("Eligible elections, stale-price calendar:", n_eligible_el, "| raw closes:", int(raw_eligible.sum()))
differ = el_rows & (raw_eligible != events["eligible"])
if differ.any():
    print(events.loc[differ, ["event_id", "source_date", "day0", "eligible_from", "eligible"]]
          .assign(raw_eligible_from=events.loc[differ, "country"].map(raw_first)).to_string(index=False))
assert n_eligible_el == 145, (
    f"{n_eligible_el} eligible elections instead of the 145 of 02_sample. If the raw-close count above is "
    "145, the stale-price rule moves eligible_from in the countries listed: stop and decide "
    "which calendar defines history before going on.")

Eligible elections, stale-price calendar: 145 | raw closes: 145


## c) Political dates
For each country: every date in `trigger_date`, `alt_trigger_dates`, `resignation_date` and
`parlgov_start_date` of `crises.csv` (all rows, coded or not) and the date of every parliamentary
election in the full `view_election` table (also before 1997, since the estimation windows of the first
events reach back into 1996). Each date is mapped to the first trading day on or after it, as day 0 is;
a date after the last close of the index has no trading day.

In [7]:
pol = []
for col in DATE_COLS:
    s = crises[["country", "event_id", col]].rename(columns={col: "date"})
    s = s.assign(date=s["date"].str.split(";")).explode("date")
    pol.append(s[s["date"] != ""].assign(source=col))
pol = pd.concat(pol)
pol["date"] = pd.to_datetime(pol["date"], format="%Y-%m-%d")

el_all = el_party.drop_duplicates("election_id")
pol = pd.concat([pol, pd.DataFrame({
    "country": el_all["country_name_short"],
    "event_id": el_all["country_name_short"] + "_E" + el_all["election_id"].astype(str),
    "date": el_all["election_date"],
    "source": "election_date"})], ignore_index=True)

pol["trading_day"] = pd.NaT
for c, idx in pol.groupby("country").groups.items():
    pos = calendar_position(calendars[c], pol.loc[idx, "date"])
    pol.loc[idx, "trading_day"] = [calendars[c][int(p)] if pd.notna(p) else pd.NaT for p in pos]
pol["trading_day"] = pd.to_datetime(pol["trading_day"])
political_dates = (pol[["country", "date", "source", "event_id", "trading_day"]]
                   .sort_values(["country", "date", "source", "event_id"]).reset_index(drop=True))

# every event's own source date is a political date of its country
own = set(zip(events["country"], events["source_date"]))
assert own <= set(zip(political_dates["country"], political_dates["date"]))
print(political_dates.groupby("source").size().to_string())
print("Political dates without a trading day (after the last close):",
      int(political_dates["trading_day"].isna().sum()))

source
alt_trigger_dates     111
election_date         636
parlgov_start_date    149
resignation_date       47
trigger_date          126
Political dates without a trading day (after the last close): 0


## e) Predictors known on day 0

**Seat shares** use, as denominator, the sum of the seats of the parties listed for that election in
`view_election` (parties with missing seats are not listed). `enp` = 1 / sum of squared seat shares.

**Left-right position** of a parliament = seat-weighted mean of `left_right` over the parties that have
a value, with the weights renormalised to sum to 1 (parties without a value are assumed to sit at the
mean of the known ones; no event is dropped). `lr_unknown_share` = seat share of the parties without a
value, the larger of the new and the old parliament. `lr_shift` (elections only) = absolute difference
between this election's parliament and the previous parliamentary election's (from the full table, also
before 1997). In the model it enters as `lr_shift_el`, which is 0 for every non-election event.

**Incumbent cabinet**: `coalition` = 1 if it has two or more parties with `cabinet_party == 1`;
`minority` = 1 if their seats are below `MAJORITY_SHARE` of `election_seats_total`. A technocratic
incumbent (no seats held by cabinet parties) gets coalition = 0, minority = 1 and `technocratic` = 1
(descriptive only).

A predictor that cannot be computed stays missing and is listed with its reason in `feature_missing`.

In [8]:
# Parliament-level quantities, for every parliamentary election of the 23 countries
listed = el_party[el_party["seats"].notna()]
seat_sum = listed.groupby("election_id")["seats"].sum()
seat_sum = seat_sum[seat_sum > 0]
listed = listed[listed["election_id"].isin(seat_sum.index)]
share = listed["seats"] / listed["election_id"].map(seat_sum)
known = listed[listed["left_right"].notna()]
parl = pd.DataFrame({
    "seat_sum": seat_sum,
    "enp": 1 / (share ** 2).groupby(listed["election_id"]).sum(),
    "lr": (known["seats"] * known["left_right"]).groupby(known["election_id"]).sum()
          / known.groupby("election_id")["seats"].sum(),
    "lr_unknown": listed["seats"].where(listed["left_right"].isna(), 0).groupby(listed["election_id"]).sum()
                  / seat_sum,
})
all_parl_ids = set(el_party["election_id"])

# Cabinet-level quantities, for every cabinet
cp = cab[cab["cabinet_party"] == 1]
gov = pd.DataFrame({   # cabinets without any cabinet party get 0 parties and 0 seats
    "n_parties": cp.groupby("cabinet_id")["party_id"].nunique(),
    "gov_seats": cp.groupby("cabinet_id")["seats"].sum(),
    "seats_missing": cp["seats"].isna().groupby(cp["cabinet_id"]).any(),
}).reindex(cab1.index, fill_value=0)
gov["seats_missing"] = gov["seats_missing"].astype(bool)
total = cab1["election_seats_total"].where(cab1["election_seats_total"] > 0)
techno = (gov["gov_seats"] == 0) & ~gov["seats_missing"]
cabinet_features = pd.DataFrame({
    "technocratic": techno.astype(float),
    "coalition": ((gov["n_parties"] >= 2) & ~techno).astype(float),
    "minority": (gov["gov_seats"] / total < MAJORITY_SHARE).astype(float)
                .where(total.notna() & ~gov["seats_missing"]).mask(techno, 1.0),
})

In [9]:
inc = events["incumbent_cabinet_id"]
inc_found = inc.isin(cab1.index)
for col in ["coalition", "minority", "technocratic"]:
    events[col] = inc.map(cabinet_features[col]).astype(float)
events["crisis"] = (events["group"] == "crisis").astype(int)
events["enp"] = events["parliament_election_id"].map(parl["enp"]).astype(float)

is_el = events["group"] == "election"
prev = events["previous_election_id"]
lr_new = events["parliament_election_id"].map(parl["lr"]).astype(float)
lr_old = prev.map(parl["lr"]).astype(float)
events["lr_shift"] = (lr_new - lr_old).abs().where(is_el)
events["lr_shift_el"] = events["lr_shift"].where(is_el, 0.0)
unk = pd.concat([events["parliament_election_id"].map(parl["lr_unknown"]).astype(float),
                 prev.map(parl["lr_unknown"]).astype(float)], axis=1)
events["lr_unknown_share"] = unk.max(axis=1, skipna=False).where(is_el)

# Reasons for missing predictors (first matching reason)
no_inc = inc.isna()
inc_reason = np.select([no_inc, ~inc_found],
                       ["previous_cabinet_id missing in ParlGov",
                        "incumbent cabinet_id not in view_cabinet"],
                       default="seats of a cabinet party or election_seats_total missing")
parl_id = events["parliament_election_id"]
enp_reason = np.select([no_inc & ~is_el, ~inc_found & ~is_el, parl_id.isna(), ~parl_id.isin(all_parl_ids)],
                       ["no incumbent cabinet (previous_cabinet_id missing in ParlGov), so no parliament",
                        "incumbent cabinet_id not in view_cabinet, so no parliament",
                        "incumbent cabinet has no election_id",
                        "parliament election not among parliamentary elections in view_election"],
                       default="no party with seats listed for that election")
lr_reason = np.select([prev.isna(), ~prev.isin(all_parl_ids), lr_new.isna() | lr_old.isna()],
                      ["previous_parliament_election_id missing in ParlGov",
                       "previous parliamentary election not in view_election",
                       "no party with both seats and left_right in one of the two parliaments"],
                      default="seats missing in one of the two parliaments")
reasons = {"coalition": inc_reason, "minority": inc_reason, "technocratic": inc_reason,
           "enp": enp_reason, "lr_shift": lr_reason, "lr_unknown_share": lr_reason}
missing_rows = []
main_rows = events["variant"] == "main"
for col, why in reasons.items():
    applies = main_rows & (is_el if col.startswith("lr_") else True)
    miss = applies & events[col].isna()
    missing_rows.append(pd.DataFrame({"event_id": events.loc[miss, "event_id"],
                                      "group": events.loc[miss, "group"],
                                      "feature": col, "reason": why[miss.to_numpy()]}))
feature_missing = pd.concat(missing_rows, ignore_index=True)
print(f"Missing predictors ({len(feature_missing)} event-feature pairs):")
print(feature_missing.to_string(index=False) if len(feature_missing) else "none")

Missing predictors (0 event-feature pairs):
none


### Checks on the predictors

In [10]:
# incumbent started before day 0 (no look-ahead)
with_inc = inc_found & events["day0"].notna()
inc_start = inc[with_inc].map(cab1["start_date"])
assert (inc_start < events.loc[with_inc, "day0"]).all(), \
    events.loc[with_inc][~(inc_start < events.loc[with_inc, "day0"])][["event_id", "day0"]]

# value ranges
assert (events["enp"].dropna() >= 1).all()
for col in ["coalition", "minority", "technocratic"]:
    assert events[col].dropna().isin([0, 1]).all(), col
assert events["lr_unknown_share"].dropna().between(0, 1).all()
assert (events.loc[~is_el, "lr_shift_el"] == 0).all()

# name check (not a join): ParlGov name of the incumbent vs previous_cabinet in crises.csv
cab_rows = events[(events["group"] != "election") & main_rows]
parlgov_name = cab_rows["incumbent_cabinet_id"].map(cab1["cabinet_name"]).astype("string").str.strip()
coded_name = cab_rows["event_id"].map(by_id["previous_cabinet"]).astype("string").str.strip()
name_diff = cab_rows.assign(parlgov_incumbent=parlgov_name, crises_previous_cabinet=coded_name)
name_diff = name_diff[parlgov_name.fillna("<none>") != coded_name.fillna("<none>")]
print("Cabinet changes where the ParlGov incumbent name differs from crises.csv previous_cabinet:")
print(name_diff[["event_id", "group", "incumbent_cabinet_id", "parlgov_incumbent", "crises_previous_cabinet"]]
      .to_string(index=False) if len(name_diff) else "none")

Cabinet changes where the ParlGov incumbent name differs from crises.csv previous_cabinet:
none


In [11]:
# elections whose listed party seats do not add up to seats_total (all parliaments used above)
el_total = el_party.drop_duplicates("election_id").set_index("election_id")
used = pd.Series(pd.concat([events["parliament_election_id"], prev]).dropna().unique())
used = used[used.isin(el_total.index)]
seat_check = pd.DataFrame({
    "country": used.map(el_total["country_name_short"]).to_numpy(),
    "election_date": used.map(el_total["election_date"]).dt.date.to_numpy(),
    "listed_seats": used.map(parl["seat_sum"]).to_numpy(),
    "seats_total": used.map(el_total["seats_total"]).to_numpy()}, index=used.to_numpy())
seat_check = seat_check[seat_check["listed_seats"] != seat_check["seats_total"]]
print("Parliaments where the sum of listed party seats differs from seats_total:")
print(seat_check.sort_values(["country", "election_date"]).to_string() if len(seat_check) else "none")

Parliaments where the sum of listed party seats differs from seats_total:
     country election_date  listed_seats  seats_total
1119     FIN    2023-04-02         199.0          200


In [12]:
# technocratic incumbents (expected: Bierlein I and II, Pikramenos, Sarmas, Monti)
tech = events[main_rows & (events["technocratic"] == 1)]
print("Technocratic incumbents:")
print(tech.assign(incumbent=tech["incumbent_cabinet_id"].map(cab1["cabinet_name"]))
      [["event_id", "group", "incumbent_cabinet_id", "incumbent"]].to_string(index=False))

# elections with a large seat share without left_right
# (expected: IRL 2011, 2016, 2020; ISR 2015; ITA 2013, 2018, 2022)
flag_lr = events[main_rows & is_el & (events["lr_unknown_share"] > LR_UNKNOWN_FLAG)]
print(f"\nElections with lr_unknown_share > {LR_UNKNOWN_FLAG}:")
print(flag_lr[["event_id", "source_date", "lr_unknown_share"]].round({"lr_unknown_share": 3})
      .to_string(index=False))

Technocratic incumbents:
 event_id    group  incumbent_cabinet_id   incumbent
AUT_E1080 election                  1600  Bierlein I
 GRC_E784 election                  1039  Pikramenos
GRC_E1123 election                  1696      Sarmas
 ITA_E789 election                   983       Monti
 AUT_1609  planned                  1608 Bierlein II

Elections with lr_unknown_share > 0.1:
 event_id source_date  lr_unknown_share
IRL_E1002  2016-02-26             0.120
IRL_E1088  2020-02-08             0.120
 ISR_E852  2015-03-17             0.108
ISR_E1047  2019-04-09             0.108
 ITA_E789  2013-02-25             0.178
ITA_E1024  2018-03-04             0.363
ITA_E1113  2022-09-25             0.363


## d) Overlapping events (main labels)
Among eligible analysis events of the same country (elections, crises, planned changes; ambiguous rows
are out of the main analysis): day 0 at most `OVERLAP_DROP` trading days apart -> the earlier is kept,
the later dropped (both listed); more than `OVERLAP_DROP` and at most `OVERLAP_FLAG` apart -> both
kept and flagged `overlap10`. Because drops, relabels and new trigger dates change which events
overlap, `apply_robustness_set` recomputes both for every set; the columns stored in `events.csv` are
the main-label result (empty for rows outside the main analysis).

In [13]:
analysis_main = main_rows & events["group"].isin(ANALYSIS_GROUPS)
drop, flag10 = overlap_flags(events[analysis_main])
events["overlap_drop"] = drop.reindex(events.index)
events["overlap10"] = flag10.reindex(events.index)

def show_overlaps(ev, title):
    print(title)
    cols = ["event_id", "group", "country", "day0", "day0_pos"]
    out = ev[ev["overlap_drop"].eq(True) | ev["overlap10"].eq(True)]
    for c, g in out.groupby("country"):
        partners = ev[(ev["country"] == c) & ev["eligible"]].sort_values("day0_pos")
        print(partners[cols + ["overlap_drop", "overlap10"]][
            partners["day0_pos"].between(g["day0_pos"].min() - OVERLAP_FLAG, g["day0_pos"].max() + OVERLAP_FLAG)]
              .to_string(index=False))
    if out.empty:
        print("none")

show_overlaps(events[analysis_main], "Overlapping events, main labels (with their neighbours):")

Overlapping events, main labels (with their neighbours):
none


## f) Robustness sets
`analysis.apply_robustness_set(events, set_name)` returns the analysis table of one set. Crisis counts
before eligibility are checked against the coding.

In [14]:
SETS = ["main"] + list(dict.fromkeys(rob["set"]))
expected_crises = {"main": 80, "R_add": 88, "R_drop": 78, "R_vol": 73, "R_bundle": 82, "R_tv": 64,
                   "R_split": 79}
assert set(SETS) == set(expected_crises), SETS
by_set = {s: apply_robustness_set(events, s) for s in SETS}
n_crises = {s: int((t["group"] == "crisis").sum()) for s, t in by_set.items()}
assert n_crises == expected_crises, n_crises

# the helper reproduces the main-label overlap stored above
m = by_set["main"].set_index("event_id")
stored = events[analysis_main].set_index("event_id")
assert (m["overlap_drop"] == stored.loc[m.index, "overlap_drop"].astype(bool)).all()
assert (m["overlap10"] == stored.loc[m.index, "overlap10"].astype(bool)).all()

for s in SETS[1:]:
    t = by_set[s]
    print(f"{s}: overlap drops {t.loc[t['overlap_drop'], 'event_id'].tolist()}, "
          f"overlap10 {t.loc[t['overlap10'], 'event_id'].tolist()}")

R_add: overlap drops [], overlap10 []
R_drop: overlap drops [], overlap10 []
R_vol: overlap drops [], overlap10 []
R_bundle: overlap drops [], overlap10 []
R_tv: overlap drops [], overlap10 []
R_split: overlap drops [], overlap10 []


In [15]:
# share of Israel and Japan among crises and among all eligible analysis events (main labels)
t = by_set["main"]
isr_jpn = t["country"].isin(["ISR", "JPN"])
for name, rows in [("crises", t["group"] == "crisis"),
                   ("eligible crises", (t["group"] == "crisis") & t["eligible"]),
                   ("eligible events", t["eligible"]),
                   ("eligible events after overlap", t["eligible"] & ~t["overlap_drop"])]:
    print(f"ISR + JPN among {name}: {int((isr_jpn & rows).sum())} of {int(rows.sum())} "
          f"({(isr_jpn & rows).sum() / rows.sum():.1%})")

ISR + JPN among crises: 23 of 80 (28.7%)
ISR + JPN among eligible crises: 23 of 76 (30.3%)
ISR + JPN among eligible events: 53 of 264 (20.1%)
ISR + JPN among eligible events after overlap: 53 of 264 (20.1%)


## g) Outputs
- `events.csv`: one row per event and variant (`main`, or the set that needs an alternative day 0).
- `political_dates.csv`: political dates of every country and their trading day.
- `sample_flow.csv`: candidates -> coded -> eligible -> after overlap, by group and set.
- `feature_missing.csv`: every missing predictor with its reason.

In [16]:
def sample_flow(set_name, table):
    rows = [("candidates", "election", len(elections)), ("candidates", "cabinet change", len(crises))]
    in_set = set(table["event_id"])
    rs = rob[rob["set"] == set_name]
    dropped = set(rs.loc[rs["action"] == "drop", "event_id"])
    for g in ANALYSIS_GROUPS:
        rows.append(("coded", g, int((table["group"] == g).sum())))
    ambiguous_out = (cr["is_crisis"] == "ambiguous") & ~cr["event_id"].isin(in_set | dropped)
    rows += [("coded", "ambiguous, not analysed", int(ambiguous_out.sum())),
             ("coded", "dropped by the set", len(dropped)),
             ("coded", "uncoded", int((~coded).sum()))]
    for g in ANALYSIS_GROUPS:
        rows.append(("eligible", g, int(((table["group"] == g) & table["eligible"]).sum())))
    for g in ANALYSIS_GROUPS:
        rows.append(("after overlap", g,
                     int(((table["group"] == g) & table["eligible"] & ~table["overlap_drop"]).sum())))
    return pd.DataFrame(rows, columns=["stage", "group", "n"]).assign(set=set_name)

flow = pd.concat([sample_flow(s, by_set[s]) for s in SETS], ignore_index=True)[["set", "stage", "group", "n"]]
print(flow.pivot_table(index=["stage", "group"], columns="set", values="n", sort=False)[SETS]
      .astype(int).to_string())

set                                    main  R_add  R_drop  R_vol  R_bundle  R_tv  R_split
stage         group                                                                       
candidates    election                  174    174     174    174       174   174      174
              cabinet change            149    149     149    149       149   149      149
coded         election                  174    174     174    174       174   174      174
              crisis                     80     88      78     73        82    64       79
              planned                    44     38      44     44        42    41       45
              ambiguous, not analysed     2      0       2      2         2     1        2
              dropped by the set          0      0       2      7         0    20        0
              uncoded                    23     23      23     23        23    23       23
eligible      election                  145    145     145    145       145   145      145

In [17]:
EVENT_COLS = ["event_id", "variant", "country", "group", "source_date", "day0", "day1", "day0_pos",
              "eligible_from", "eligible", "ineligible_reason", "overlap_drop", "overlap10",
              "crisis", "coalition", "minority", "technocratic", "enp", "lr_shift", "lr_shift_el",
              "lr_unknown_share", "incumbent_cabinet_id", "parliament_election_id", "election_id",
              "parlgov_cabinet_id"]
events_out = events[EVENT_COLS].sort_values(["country", "source_date", "event_id", "variant"])
for name, df in [("events", events_out), ("political_dates", political_dates),
                 ("sample_flow", flow), ("feature_missing", feature_missing)]:
    print(save_derived(df, name), len(df), "rows")

/files/mywork/elections_project/data/derived/events.csv 302 rows
/files/mywork/elections_project/data/derived/political_dates.csv 1069 rows
/files/mywork/elections_project/data/derived/sample_flow.csv 98 rows
/files/mywork/elections_project/data/derived/feature_missing.csv 0 rows
